In [2]:
import pandas as pd
import numpy as np

fy2014 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/H-1B_FY14_Q4.xlsx"

df = pd.read_excel(fy2014)

print(df.shape)
print(df.head())

(519504, 35)
      LCA_CASE_NUMBER               STATUS     LCA_CASE_SUBMIT  \
0  I-200-09106-546545  CERTIFIED-WITHDRAWN 2013-01-18 18:05:43   
1  I-200-09162-143337            CERTIFIED 2014-03-17 11:11:45   
2  I-200-09182-550211               DENIED 2014-03-14 17:05:01   
3  I-200-09193-873099            CERTIFIED 2014-03-01 12:37:02   
4  I-200-09197-120609  CERTIFIED-WITHDRAWN 2014-03-10 15:33:41   

        DECISION_DATE VISA_CLASS LCA_CASE_EMPLOYMENT_START_DATE  \
0 2014-09-29 15:59:50       H-1B                     2013-02-01   
1 2014-03-21 23:11:51       H-1B                     2014-09-15   
2 2014-03-20 16:59:49       H-1B                     2014-04-01   
3 2014-03-06 22:03:37       H-1B                     2014-08-28   
4 2014-03-17 10:52:46       H-1B                     2014-09-01   

  LCA_CASE_EMPLOYMENT_END_DATE             LCA_CASE_EMPLOYER_NAME  \
0                   2016-02-01  TEXAS STATE UNIVERSITY-SAN MARCOS   
1                   2017-09-15   EYE SPECIALISTS 

In [4]:
print(df.columns.tolist())

['LCA_CASE_NUMBER', 'STATUS', 'LCA_CASE_SUBMIT', 'DECISION_DATE', 'VISA_CLASS', 'LCA_CASE_EMPLOYMENT_START_DATE', 'LCA_CASE_EMPLOYMENT_END_DATE', 'LCA_CASE_EMPLOYER_NAME', 'LCA_CASE_EMPLOYER_ADDRESS', 'LCA_CASE_EMPLOYER_CITY', 'LCA_CASE_EMPLOYER_STATE', 'LCA_CASE_EMPLOYER_POSTAL_CODE', 'LCA_CASE_SOC_CODE', 'LCA_CASE_SOC_NAME', 'LCA_CASE_JOB_TITLE', 'LCA_CASE_WAGE_RATE_FROM', 'LCA_CASE_WAGE_RATE_TO', 'LCA_CASE_WAGE_RATE_UNIT', 'FULL_TIME_POS', 'TOTAL_WORKERS', 'LCA_CASE_WORKLOC1_CITY', 'LCA_CASE_WORKLOC1_STATE', 'PW_1', 'PW_UNIT_1', 'PW_SOURCE_1', 'OTHER_WAGE_SOURCE_1', 'YR_SOURCE_PUB_1', 'LCA_CASE_WORKLOC2_CITY', 'LCA_CASE_WORKLOC2_STATE', 'PW_2', 'PW_UNIT_2', 'PW_SOURCE_2', 'OTHER_WAGE_SOURCE_2', 'YR_SOURCE_PUB_2', 'LCA_CASE_NAICS_CODE']


In [13]:
import pandas as pd

df = pd.read_excel(fy2014)

df["LCA_CASE_EMPLOYER_NAME"] = (
    df["LCA_CASE_EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["LCA_CASE_EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "LCA_CASE_EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

LCA_CASE_EMPLOYER_NAME
MICROSOFT CORPORATION                                 3750
GOOGLE INC.                                           2949
AMAZON CORPORATE LLC                                  1596
APPLE INC.                                            1458
FACEBOOK, INC.                                         778
MICRON TECHNOLOGY, INC.                                242
ADVANCED MICRO DEVICES, INC.                           236
TESLA MOTORS, INC.                                     226
NVIDIA CORPORATION                                     167
GOOGLE, INC.                                           114
AMAZON WEB SERVICES, INC.                               88
APPLEXUS TECHNOLOGIES LLC                               62
APPLET SYSTEMS LLC                                      27
AMAZON FULFILLMENT SERVICES, INC.                       24
AMAZON.COM.DEDC, LLC                                    21
METANOIA SOLUTIONS INC                                  18
KENNAMETAL, INC.                 

In [15]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["LCA_CASE_EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Microsoft    3750
Apple        1458
Meta          778
Micron        242
AMD           236
Tesla         226
NVIDIA        167
Name: count, dtype: int64


In [17]:
df_clean = df.copy()

In [19]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["LCA_CASE_WORKLOC1_CITY"] = (
    df["LCA_CASE_WORKLOC1_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["LCA_CASE_WORKLOC1_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Austin      152
Seattle      78
New York     72
San Jose     10
Name: count, dtype: int64


In [25]:
print(df["LCA_CASE_WAGE_RATE_UNIT"].value_counts())

LCA_CASE_WAGE_RATE_UNIT
Year    312
Name: count, dtype: int64


In [28]:
import pandas as pd

# Convert wage values to numbers
df["LCA_CASE_WAGE_RATE_FROM"] = pd.to_numeric(
    df["LCA_CASE_WAGE_RATE_FROM"],
    errors="coerce"
)

# Standardize wage units
df["LCA_CASE_WAGE_RATE_UNIT"] = (
    df["LCA_CASE_WAGE_RATE_UNIT"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["LCA_CASE_WAGE_RATE_FROM"]
    * df["LCA_CASE_WAGE_RATE_UNIT"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]].head(10))

      Company     City  LCA_CASE_WAGE_RATE_FROM LCA_CASE_WAGE_RATE_UNIT  \
128       AMD   Austin                  66123.0                    YEAR   
1508      AMD   Austin                  71282.0                    YEAR   
2372      AMD   Austin                  74372.0                    YEAR   
3104   NVIDIA   Austin                  82400.0                    YEAR   
3298      AMD   Austin                  89606.0                    YEAR   
7092      AMD   Austin                  90000.0                    YEAR   
7833      AMD   Austin                  93413.0                    YEAR   
9120      AMD   Austin                  74464.0                    YEAR   
10096    Meta  Seattle                 133000.0                    YEAR   
10502     AMD   Austin                  89606.0                    YEAR   

       Annual_Wage  
128        66123.0  
1508       71282.0  
2372       74372.0  
3104       82400.0  
3298       89606.0  
7092       90000.0  
7833       93413.0  
9120  

In [30]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count       312.000000
mean     110032.412949
std       29396.751556
min       62004.000000
25%       84784.750000
50%      105290.500000
75%      130000.000000
max      210000.000000
Name: Annual_Wage, dtype: float64


In [32]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [34]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage      Mean_Wage
0     Austin        AMD                 94    82430.000   86752.468085
1     Austin      Apple                 40    90000.000   91737.325000
2     Austin       Meta                  2    82957.225   82957.225000
3     Austin     NVIDIA                 16    84739.000   92006.812500
4   New York      Apple                  3   130000.000  118966.666667
5   New York       Meta                 55   130000.000  133632.029818
6   New York  Microsoft                 14   145396.000  141945.857143
7   San Jose     Micron                  9   110000.000  114772.777778
8   San Jose  Microsoft                  1   130000.000  130000.000000
9    Seattle       Meta                 62   130000.000  134222.447581
10   Seattle  Microsoft                 15   107000.000  104862.800000
11   Seattle     NVIDIA                  1   116272.000  116272.000000


In [38]:
columns_to_keep = [
    "Company",
    "LCA_CASE_EMPLOYER_NAME",
    "LCA_CASE_JOB_TITLE",
    "LCA_CASE_SOC_NAME",
    "City",
    "LCA_CASE_WAGE_RATE_FROM",
    "LCA_CASE_WAGE_RATE_UNIT",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2014.csv",
    index=False
)